In [53]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage , HumanMessage
from typing import TypedDict , Annotated
from langgraph.graph.message import add_messages # reduser for BaseMessage same as operater.app (+)
from langgraph.checkpoint.memory import MemorySaver # this saves in RAM 

In [54]:
class chatState(TypedDict):

    message: Annotated[list[BaseMessage], add_messages]

In [55]:
llm = ChatOllama(model="qwen2.5:0.5b")

In [56]:
def chat_node(state: chatState):

    message= state['message']

    responce= llm.invoke(message)

    return {'message': [responce]}

In [60]:
checkpoint=MemorySaver()

graph=StateGraph(chatState)

#nodes
graph.add_node('chat_node',chat_node)

# edges 
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatBot=graph.compile(checkpointer=checkpoint)

In [58]:
initial_state={'message':[HumanMessage(content="what is fullform of AI")]}

response=chatBot.invoke(initial_state)

In [59]:
response['message'][-1]

AIMessage(content='AI stands for Artificial Intelligence. In full form, it is abbreviated as AI or AI. AI is the artificial intelligence that refers to the use of computers and machines to perform tasks that typically require human intelligence. It involves the development of algorithms, machine learning, natural language processing, and other techniques that enable machines to think, learn, and solve problems like humans do. AI is a rapidly evolving field with applications in various industries, including healthcare, finance, transportation, and entertainment.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:0.5b', 'created_at': '2026-10-22T07:11:38.0946683Z', 'done': True, 'done_reason': 'stop', 'total_duration': 707688400, 'load_duration': 18794600, 'prompt_eval_count': 35, 'prompt_eval_duration': 82081000, 'eval_count': 96, 'eval_duration': 595398000, 'logprobs': None, 'model_name': 'qwen2.5:0.5b', 'model_provider': 'ollama'}, id='lc_run--01a16273-79aa-7752-b722-89f6fe4

In [61]:
thread_id='1'

while True:

    user_message=input("Enter prompt")
    
    print(f"user prompt : {user_message}")

    if user_message.strip().lower() in ['bye','quit','exit']:
        break

    config= {'configurable' : {'thread_id':thread_id}}
    response= chatBot.invoke({'message': HumanMessage(content=user_message)},config=config)

    print(response['message'][-1].content)

user prompt : hi
Hello! How can I help you today?
user prompt : my name is prit
Hello! How can I assist you with your name?
user prompt : what is my name
Your name is "Prit". It's a combination of two words: "Prit" and "rit". Prit is a type of clothing worn by men, while rit is a type of music genre.
user prompt : bye
